# Notebook clean pour les étapes d'Extract & Transform

---
## 1 - EXTRACT

### Chargement des dépendances

Ce bloc charge les variables d'environnement depuis un fichier `.env` et importe les bibliothèques essentielles pour le notebook :

- **Manipulation de données** : `pandas`, `numpy`
- **Gestion de fichiers/config** : `os`, `yaml`, `dotenv`, `Path`
- **Requêtes HTTP** : `requests`
- **Base de données SQL** : `sqlalchemy`, `text`, `SQLAlchemyError`
- **Structures de données** : `dataclasses`

In [ ]:
%load_ext autoreload
%autoreload 2

# Charger les dépendances

import os
import yaml
import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
from dotenv import load_dotenv
from pathlib import Path
from dataclasses import dataclass

load_dotenv()

---
### Configuration et chargement des données


1. **Chemin racine**
   Définit `ROOT_DIR` comme le dossier parent du notebook (ou le dossier courant si le script n'est pas exécuté directement).
   Affiche le chemin vers `config.yml`.

2. **Fonction `load_config`**
   - Lit un fichier YAML.
   - Remplace les valeurs au format `${VAR}` par les variables d'environnement correspondantes.

3. **Chargement de la configuration**
   - Charge `config.yml` dans `conf`.
   - Extrait la section `database` dans `db_conf`.

4. **Chemin du fichier CSV**
   Définit le chemin vers `acc_2017.csv` dans le dossier `data` et l'affiche.


In [ ]:
try:
    ROOT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    ROOT_DIR = Path.cwd().parent

CONFIG_PATH = ROOT_DIR / "config.yml"
print(CONFIG_PATH)

In [ ]:
def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        
    for section, values in config.items():
        for key, val in values.items():
            if isinstance(val, str) and val.startswith("${"):
                env_var = val.strip("${}")
                config[section][key] = os.getenv(env_var)    
    return config

In [ ]:
conf = load_config(CONFIG_PATH)
db_conf = conf['database']

In [ ]:
csv = ROOT_DIR / "data" / "acc_2017.csv"
print(csv)

---
### Téléchargement du CSV

Ce script télécharge le dataset **accidents corporels** depuis [OpenDataSoft](https://public.opendatasoft.com) et le sauvegarde en local.

#### Fonctionnement :
- **Configuration** : Utilise une classe `Config` pour définir l'URL, le nom du fichier, et les paramètres de téléchargement (taille des chunks, timeout, etc.).
- **Téléchargement** :
  - Construit l'URL de l'API.
  - Télécharge le fichier **par chunks** (64 Ko) pour éviter de surcharger la mémoire.
  - Sauvegarde le résultat dans `data/accidents_corporels_millesime.csv`.
- **Exécution** : Appelle `main()` pour lancer le téléchargement.


In [ ]:
"""Téléchargement minimaliste du dataset accidents corporels depuis OpenDataSoft.

Version simplifiée sans retry, sans barre de progression, sans validation.
Télécharge le CSV par chunks et le sauvegarde dans data/accidents_corporels_millesime.csv

Usage:
    python scripts/sauvegarde_csv_api_v1.py

Source:
    https://public.opendatasoft.com - Dataset accidents corporels de la circulation
"""


@dataclass(frozen=True)
class Config:
    """Configuration du téléchargement."""
    
    base_url: str = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets"
    dataset_id: str = "accidents-corporels-de-la-circulation-millesime"
    output_dir: str = "data"
    output_file: str = "accidents_corporels_millesime.csv"
    delimiter: str = ","
    chunk_size: int = 65536 #on lit 64KB par 64KB pour ne pas que Lounes voit la RAM de son pc bruler
    timeout: int = 30


def build_url(config: Config) -> str:
    """Construit l'URL de téléchargement."""
    return f"{config.base_url}/{config.dataset_id}/exports/csv?delimiter={config.delimiter}"


def resolve_path(config: Config) -> Path:
    """Détermine le chemin de sortie."""
    #script_dir = ROOT_DIR
    return ROOT_DIR / config.output_dir / config.output_file


def download_csv(url: str, destination: Path, config: Config) -> None:
    """Télécharge le CSV par chunks."""
    print("Téléchargement depuis OpenDataSoft...")

    response = requests.get(url, stream=True, timeout=config.timeout)
    response.raise_for_status()

    destination.parent.mkdir(parents=True, exist_ok=True)

    with response, open(destination, "wb") as handle:
        for chunk in response.iter_content(chunk_size=config.chunk_size):
            if chunk:
                handle.write(chunk)

    print(f"Fichier sauvegardé: {destination}")


def main() -> None:
    """Point d'entrée principal."""
    config = Config()
    url = build_url(config)
    path = resolve_path(config)

    download_csv(url, path, config)

    print("Téléchargement terminé")


if __name__ == "__main__":
    main()


---
### Importation des données

Ce bloc charge le fichier CSV téléchargé dans un **DataFrame pandas** :

- **Source** : `data/accidents_corporels_millesime.csv`
- **Paramètres** :
  - Séparateur : `,`
  - Désactive `low_memory` pour éviter les avertissements de type mixte.
- **Affichage** : Affiche les **5 premières lignes** du DataFrame avec `df_source.head()`.


In [ ]:
# Importation des données source dans un DataFrame pandas

df_source = pd.read_csv(ROOT_DIR / 'data/accidents_corporels_millesime.csv', delimiter=',', low_memory=False)
df_source.head()

In [ ]:
df_source.info()

---
## 2 - TRANSFORM

### Nettoyage et découpage des données

Cette fonction **nettoie** et **découpe** le DataFrame source en 5 DataFrames thématiques :

#### Étapes :
1. **Nettoyage préliminaire DataFrame source** :
   - Met en minuscules et supprime les espaces des noms de colonnes.
   - Convertit la colonne `datetime` en format datetime.

2. **Découpage** :
   Crée 5 DataFrames distincts :
   - **Accidents** : Caractéristiques générales de l’accident.
   - **Lieux** : Localisation et détails géographiques.
   - **Date_accident** : Informations temporelles (date, heure).
   - **Véhicules** : Détails sur les véhicules impliqués.
   - **Usagers** : Informations sur les personnes impliquées.

3. **Résultat** :
   Retourne un message de confirmation et affiche la taille de chaque DataFrame.


In [ ]:
def preprocess_and_split(df_source):
    """
    Clean the source dataframe and split it into 5 separate dataframes:
    accidents, lieux, date_accident, vehicules, usagers.
    
    Parameters:
        df_source (pd.DataFrame): Raw input dataframe.
        
    Returns:
        dict: A dictionary containing the 5 dataframes.
    """
    
    # Tell Python these variables are global
    global df_accidents, df_lieux, df_date_accident, df_vehicules, df_usagers

    # Step 1: Log starting
    print("Step 1: Starting preprocessing...")

    # Clean column names
    df_source.columns = df_source.columns.str.lower().str.strip()
    print("Columns lowercased and stripped.")
    
    # Convert datetime
    df_source["datetime"] = pd.to_datetime(df_source["datetime"], errors="coerce")
    print("Datetime column converted.")
    
    # Step 2: Create 5 separate dataframes
    print("Step 2: Splitting into 5 source dataframes...")

    df_accidents = df_source[[
        'num_acc', 'lum', 'agg', 'int', 'atm', 'adr', 'col', 'circ', 'plan', 
        'prof', 'surf', 'infra', 'situ', 'year_georef'
    ]].copy()
    print("Accidents dataframe created:", df_accidents.shape)

    df_lieux = df_source[[
        'com_code', 'com_name', 'dep_code', 'dep_name', 'reg_code', 'reg_name', 
        'epci_code', 'epci_name', 'lat', 'long', 'catr', 'v1', 'voie', 'v2', 
        'nbv', 'vosp', 'pr', 'pr1', 'lartpc', 'larrout', 'num_acc'
    ]].copy()
    print("Lieux dataframe created:", df_lieux.shape)

    df_date_accident = df_source[[
        'datetime', 'an', 'mois', 'jour', 'hrmn', 'num_acc'
    ]].copy()
    print("Date_accident dataframe created:", df_date_accident.shape)

    df_vehicules = df_source[[
        'num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv', 'num_acc'
    ]].copy()
    print("Vehicules dataframe created:", df_vehicules.shape)

    df_usagers = df_source[[
        'sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 
        'locp', 'actp', 'etatp', 'num_acc'
    ]].copy()
    print("Usagers dataframe created:", df_usagers.shape)
    
    print("Step 3: All 5 dataframes created and ready for transformation.")
    
    # Return just a simple message instead of full dataframes
    return "✅ Preprocessing done. 5 dataframes are created and ready for transformation."

In [ ]:
preprocess_and_split(df_source)

---
### Transformation des données "accidents"

Cette fonction nettoie et transforme le DataFrame `df_accidents` :

#### Actions :
1. **Remplacement des valeurs** :
   - Remplace les `-1` par `NaN` dans les colonnes catégorielles (`lum`, `agg`, `int`, `atm`, etc.).

2. **Nettoyage des colonnes** :
   - Pour `infra` et `situ`, convertit les valeurs numériques en `NaN`.

3. **Résultat** :
   - Crée un nouveau DataFrame `df_accidents_cleaned`.
   - Affiche un résumé des transformations et les informations du DataFrame final.


In [ ]:
def transform_accidents():
    """
    Transform df_accidents and create df_accidents_cleaned.
    - Replace '-1' with NaN for categorical columns
    - Clean 'infra' and 'situ' columns
    Logs each step.
    """
    
    global df_accidents, df_accidents_cleaned
    
    print("🔹 Starting transformation on df_accidents...")
    
    # Copy the original dataframe
    df_accidents_cleaned = df_accidents.copy()
    
    cat_cols = ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']
    
    # Step 1: Replace '-1' with NaN
    for col in cat_cols:
        df_accidents_cleaned[col] = df_accidents_cleaned[col].replace('-1', np.nan)
    print(f"Step 1: Replaced '-1' with NaN for columns: {cat_cols}")
    
    # Step 2: Clean 'infra' and 'situ' columns
    df_accidents_cleaned['infra'] = df_accidents_cleaned['infra'].apply(lambda x: np.nan if str(x).isdigit() else x)
    df_accidents_cleaned['situ'] = df_accidents_cleaned['situ'].apply(lambda x: np.nan if str(x).isdigit() else x)
    print("Step 2: Cleaned 'infra' and 'situ' columns (numbers converted to NaN if present)")
   
    print("✅ df_accidents_cleaned transformation completed.\n")
    print(df_accidents_cleaned.info())

In [ ]:
transform_accidents()

---
### Transformation des données "lieux"

Cette fonction nettoie et transforme le DataFrame `df_lieux` :

#### Étapes :
1. **Suppression de colonnes** :
   - Retire les colonnes jugées inutiles (`voie`, `v1`, `v2`, `lat`, `long`, etc.).

2. **Nettoyage des colonnes** :
   - **`catr`** : Convertit les valeurs numériques en `'autre'`.
   - **`nbv`** : Remplace les valeurs > 6, = 0 ou = -1 par `NaN` et convertit en `Int64`.
   - **`vosp`** : Remplace les `-1` par `NaN`.

3. **Ajout d'un identifiant** :
   - Crée une colonne `lieu_id` pour identifier chaque lieu.

4. **Résultat** :
   - Réorganise les colonnes pour placer `lieu_id` en premier.
   - Affiche un résumé des transformations et les informations du DataFrame final.


In [ ]:
def transform_lieux():
    """
    Transform df_lieux and create df_lieux_cleaned.
    - Drop unnecessary columns
    - Clean 'catr', 'nbv', and 'vosp' columns
    Logs each step.
    """
    
    global df_lieux, df_lieux_cleaned
    
    print("🔹 Starting transformation on df_lieux...")
    
    # Copy original dataframe
    df_lieux_cleaned = df_lieux.copy()
    
    # Step 1: Drop unnecessary columns
    drop_cols = ['voie', 'v1', 'v2', 'pr', 'pr1', 'lartpc', 'larrout', 
                 'epci_code', 'epci_name', 'lat', 'long']
    df_lieux_cleaned = df_lieux_cleaned.drop(columns=drop_cols)
    print(f"Step 1: Dropped columns: {drop_cols}")
    
    # Step 2: Clean 'catr'
    df_lieux_cleaned['catr'] = df_lieux_cleaned['catr'].apply(lambda x: 'autre' if str(x).isdigit() else x)
    print("Step 2: Cleaned 'catr' column (numbers converted to 'autre')")
    
    # Step 3: Clean 'nbv'
    df_lieux_cleaned['nbv'] = df_lieux_cleaned['nbv'].apply(
        lambda x: np.nan if (x > 6.0 or x == 0.0 or x == -1.0) else x
    )
    df_lieux_cleaned['nbv'] = df_lieux_cleaned['nbv'].astype('Int64')
    print("Step 3: Cleaned 'nbv' column and converted dtype to Int64")
    
    # Step 4: Clean 'vosp'
    df_lieux_cleaned['vosp'] = df_lieux_cleaned['vosp'].apply(lambda x: np.nan if x == '-1' else x)
    print("Step 4: Cleaned 'vosp' column ('-1' replaced with NaN)")

    # Step 5: Add a lieu_id column
    df_lieux_cleaned = df_lieux_cleaned.reset_index(drop=True)
    df_lieux_cleaned['lieu_id'] = df_lieux_cleaned.index + 1

    # Step 5: Reorder columns to put ID first
    cols = ['lieu_id'] + [c for c in df_lieux_cleaned.columns if c != 'lieu_id']
    df_lieux_cleaned = df_lieux_cleaned[cols]
    
    print("✅ df_lieux_cleaned transformation completed.\n")
    print(df_lieux_cleaned.info())


In [ ]:
transform_lieux()

---
### Transformation des données "date_accident"

Cette fonction nettoie et transforme le DataFrame `df_date_accident` :

#### Étapes :
1. **Conversion de la colonne `hrmn`** :
   - Convertit la colonne `hrmn` (format `HH:MM`) en objet `datetime.time`.

2. **Ajout d'un identifiant** :
   - Crée une colonne `date_accident_id` pour identifier chaque entrée.

3. **Réorganisation** :
   - Place la colonne `date_accident_id` en première position.

4. **Résultat** :
   - Affiche un résumé des transformations et les informations du DataFrame final.


In [ ]:
def transform_date_accident():
    """
    Transform df_date_accident and create df_date_accident_cleaned.
    - Convert 'hrmn' column from string/object to datetime.time
    Logs each step.
    """
    
    global df_date_accident, df_date_accident_cleaned
    
    print("🔹 Starting transformation on df_date_accident...")
    
    # Copy the original dataframe
    df_date_accident_cleaned = df_date_accident.copy()
    
    # Step 1: Convert 'hrmn' to datetime.time
    df_date_accident_cleaned['hrmn'] = pd.to_datetime(
        df_date_accident_cleaned['hrmn'], format='%H:%M', errors='coerce'
    ).dt.time
    print("Step 1: Converted 'hrmn' column to datetime.time")

    # Step 2: Add a date_accident_id column
    df_date_accident_cleaned = df_date_accident_cleaned.reset_index(drop=True)
    df_date_accident_cleaned['date_accident_id'] = df_date_accident_cleaned.index + 1

    # Step 2: Reorder columns to put ID first
    cols = ['date_accident_id'] + [c for c in df_date_accident_cleaned.columns if c != 'date_accident_id']
    df_date_accident_cleaned = df_date_accident_cleaned[cols]
    
    
    print("✅ df_date_accident_cleaned transformation completed.\n")
    print(df_date_accident_cleaned.info())


In [ ]:
transform_date_accident()

---
### Fonction d'explosion des DataFrames

Cette fonction **décompose** un DataFrame dont certaines colonnes contiennent des **valeurs séparées par des virgules**. Elle est conçue pour être utilisée sur les DataFrames `vehicules` et `usagers`.

#### Étapes clés :
1. **Conversion en listes** :
   - Transforme les valeurs séparées par des virgules en listes.

2. **Alignement des listes** :
   - Détecte et corrige les lignes où les listes n'ont pas la même longueur (remplissage avec `None`).

3. **Explosion des lignes** :
   - Crée une nouvelle ligne pour chaque élément des listes, en conservant l'alignement.

4. **Ajout d'un identifiant** :
   - Ajoute une colonne d'identifiant unique (`vehicule_id`, `usager_id`, etc.) en première position.

#### Résultat :
- Retourne un DataFrame **explosé** avec un identifiant séquentiel, prêt pour une analyse détaillée.


In [ ]:
# FONCTION D'EXPLOSION - VEHICULES ET USAGERS

def explode_df(df, cols_to_explode, id_col_name):
    """
    Explodes a DataFrame where some columns contain comma-separated values.
    Handles list alignment, padding, and adds a sequential unique ID column.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.
    cols_to_explode : list
        List of column names that contain comma-separated or list values.
    id_col_name : str
        Name of the ID column to create (e.g. 'vehicule_id', 'usager_id').
    
    Returns
    -------
    pd.DataFrame
        Exploded DataFrame with a unique sequential ID as the first column.
    """
    df = df.copy()

    # 1️⃣ Ensure all values are lists (split by comma if needed)
    for col in cols_to_explode:
        df[col] = df[col].astype(str).apply(lambda x: x.split(',') if ',' in x else [x])

    # 2️⃣ Compute length consistency per row
    list_lengths_df = df[cols_to_explode].map(lambda x: len(x) if isinstance(x, list) else 1)
    df['nunique_lengths'] = list_lengths_df.nunique(axis=1)
    df['max_length'] = list_lengths_df.max(axis=1)

    # 3️⃣ Detect misaligned rows
    misaligned_rows = df[df['nunique_lengths'] > 1]
    if len(misaligned_rows) > 0:
        print(f"⚠️ {len(misaligned_rows)} rows with misaligned list lengths detected — they will be padded.")

    # 4️⃣ Pad lists to same length
    def pad_lists(row):
        max_len = row['max_length']
        for col in cols_to_explode:
            vals = row[col] if isinstance(row[col], list) else [row[col]]
            row[col] = (vals + [None] * (max_len - len(vals)))[:max_len]
        return row

    df = df.apply(pad_lists, axis=1)

    # 5️⃣ Explode all relevant columns together
    df_exploded = df.explode(cols_to_explode, ignore_index=True)
    print(f"✓ Data exploded successfully: {len(df_exploded):,} rows.")

    # 6️⃣ Add sequential unique ID
    df_exploded = df_exploded.reset_index(drop=True)
    df_exploded[id_col_name] = df_exploded.index + 1

    # 7️⃣ Reorder columns to put ID first
    cols = [id_col_name] + [c for c in df_exploded.columns if c != id_col_name]
    df_exploded = df_exploded[cols]

    return df_exploded

### Transformation des données "véhicules"

Cette fonction nettoie et transforme le DataFrame `df_vehicules` :

#### Étapes :
1. **Explosion des colonnes** :
   - Supprime la colonne `senc`. (pas besoin pour l'analyse)
   - Décompose les colonnes contenant des listes ou valeurs séparées par des virgules (`num_veh`, `catv`, `choc`, etc.) en utilisant la fonction définie plus haut `explode_df`.

2. **Nettoyage de la colonne `catv`** :
   - Standardise les valeurs (ex : regroupe les véhicules "autres", simplifie les libellés des quads et voiturettes).

3. **Nettoyage des colonnes catégorielles** :
   - Remplace les valeurs `-1`, `nan`, ou `None` par `NaN` pour les colonnes `choc`, `obs`, `obsm`, `occutc`, et `manv`.

4. **Ajustement des types** :
   - Convertit les valeurs numériques non pertinentes en `NaN` pour `obs` et `manv`.
   - Convertit `occutc` en type `Int64`.

5. **Suppression des colonnes temporaires** :
   - Retire les colonnes `nunique_lengths` et `max_length` (utilisées pour l'explosion).

6. **Résultat** :
   - Affiche un résumé des transformations et les informations du DataFrame final `df_vehicules_cleaned`.


In [ ]:
def transform_vehicules():
    """
    Transform df_vehicules and create df_vehicules_cleaned.
    - Explode columns with comma-separated values
    - Clean categorical columns ('catv', 'choc', 'obs', 'obsm', 'occutc', 'manv')
    - Adjust dtypes
    Logs each step.
    """
    
    global df_vehicules, df_vehicules_cleaned
    
    print("🔹 Starting transformation on df_vehicules...")
    
    # Step 0: Copy original df
    df_veh_cleaned = df_vehicules.copy()
    
    # Step 1: Drop 'senc' and explode relevant columns
    df_veh_cleaned = df_veh_cleaned.drop(columns=['senc'])
    cols_to_explode = ['num_veh', 'catv', 'choc', 'obs', 'obsm', 'occutc', 'manv']
    df_veh_exploded = explode_df(df_veh_cleaned, cols_to_explode, id_col_name='vehicule_id')
    print(f"Step 1: Exploded columns: {cols_to_explode}")
    
    # Step 2: Clean 'catv' column
    def clean_catv(value):
        if pd.isna(value):
            return value
        if value == '0':
            return 'Autre véhicule'
        if value in ['50', '43', '42', '41', '60', '80']:
            return 'Autre véhicule'
        if 'Voiturette (Quadricycle à moteur carrossé) (anciennement "voiturette ou tricycle à moteur")' in value:
            return 'Voiturette'
        if 'Quad léger <= 50 cm3 (Quadricycle à moteur non carrossé)' in value:
            return 'Quad léger <= 50 cm3'
        if 'Quad lourd > 50 cm3 (Quadricycle à moteur non carrossé)' in value:
            return 'Quad lourd > 50 cm3'
        return value

    df_veh_exploded['catv'] = df_veh_exploded['catv'].apply(clean_catv)
    print("Step 2: Cleaned 'catv' column")
    
    # Step 3: Clean generic columns
    def clean_generic(value):
        if pd.isna(value) or value in ['nan', 'None']:
            return np.nan
        if value == '-1':
            return np.nan
        return value

    for col in ['choc','obs', 'obsm', 'occutc', 'manv']:
        df_veh_exploded[col] = df_veh_exploded[col].apply(clean_generic)
    print("Step 3: Cleaned 'choc', 'obs', 'obsm', 'occutc', 'manv' columns")
    
    # Step 4: Adjust dtypes
    for cols in ['obs', 'manv']:
        df_veh_exploded[cols] = df_veh_exploded[cols].apply(lambda x: np.nan if str(x).isdigit() else x)
    df_veh_exploded['occutc'] = df_veh_exploded['occutc'].astype('Int64')
    print("Step 4: Adjusted dtypes for 'obs', 'manv', and 'occutc'")
    
    # Step 5: Drop helper columns from explode_df
    df_veh_exploded = df_veh_exploded.drop(columns=['nunique_lengths', 'max_length'])
    print("Step 5: Dropped helper columns from explode_df")
    
    # Step 6: Assign to global cleaned df
    df_vehicules_cleaned = df_veh_exploded
    print("✅ df_vehicules_cleaned transformation completed.\n")
    print(df_vehicules_cleaned.info())


In [ ]:
transform_vehicules()

---
### Transformation des données "usagers"

Cette fonction nettoie et transforme le DataFrame `df_usagers` :

#### Étapes :
1. **Explosion des colonnes** :
   - Décompose les colonnes contenant des listes ou valeurs séparées par des virgules (`sexe`, `grav`, `trajet`, etc.) en utilisant `explode_df`.

2. **Normalisation des chaînes de caractères** :
   - Supprime les espaces superflus dans toutes les colonnes de type texte.

3. **Remplacement des valeurs manquantes** :
   - Remplace les chaînes `'nan'`, `'NaN'`, `'None'`, et `-1` par `NaN`.

4. **Nettoyage des colonnes `locp` et `actp`** :
   - Convertit les valeurs numériques en `NaN`.
   - Remplace les valeurs `'A'` et `'B'` par `NaN` pour `actp`.

5. **Gestion des colonnes spécifiques aux piétons** :
   - Met à `NaN` les colonnes `locp`, `actp`, et `etatp` si aucun piéton n'est impliqué dans l'accident.

6. **Suppression des colonnes temporaires** :
   - Retire les colonnes `nunique_lengths` et `max_length` (utilisées pour l'explosion).

7. **Résultat** :
   - Affiche un résumé des transformations et les informations du DataFrame final `df_usagers_cleaned`.


In [ ]:
def transform_usagers():
    """
    Transform df_usagers and create df_usagers_cleaned.
    - Explode columns with comma-separated values using explode_df
    - Apply cleaning logic directly
    - Drop helper columns
    Logs each step.
    """
    
    global df_usagers, df_usagers_cleaned
    
    print("🔹 Starting transformation on df_usagers...")
    
    # Step 0: Copy original
    df_us_cleaned = df_usagers.copy()
    
    # Step 1: Explode relevant columns
    cols_to_explode = ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 
                       'catu', 'place', 'locp', 'actp', 'etatp']
    df_usager_exploded = explode_df(df_us_cleaned, cols_to_explode, id_col_name='usager_id')
    print(f"Step 1: Exploded columns: {cols_to_explode}")
    
    # Step 2: Normalize strings and strip whitespace
    str_cols = df_usager_exploded.select_dtypes(include="object").columns
    for col in str_cols:
        df_usager_exploded[col] = df_usager_exploded[col].astype(str).str.strip()
    print("Step 2: Normalized string columns (stripped whitespace)")
    
    # Step 3: Replace string missing values with np.nan
    df_usager_exploded.replace(to_replace=['nan', 'NaN', 'None', '-1'], value=np.nan, inplace=True)
    print("Step 3: Replaced 'nan', 'NaN', 'None', '-1' with np.nan")
    
    # Step 4: Clean 'locp' and 'actp'
    df_usager_exploded['locp'] = df_usager_exploded['locp'].replace(r'^\d+$', np.nan, regex=True)
    df_usager_exploded['actp'] = df_usager_exploded['actp'].replace(r'^\d+$', np.nan, regex=True)
    df_usager_exploded['actp'] = df_usager_exploded['actp'].replace(['A', 'B'], np.nan)
    print("Step 4: Cleaned 'locp' and 'actp' columns")
    
    # Step 5: Nullify pedestrian-only columns where no pedestrian in accident
    mask_pieton_present = df_usager_exploded.groupby("num_acc")["catu"].transform(lambda x: (x == "Piéton").any())
    cols_to_null = ["locp", "actp", "etatp"]
    df_usager_exploded.loc[~mask_pieton_present, cols_to_null] = np.nan
    print("Step 5: Nullified pedestrian-only columns where no pedestrian in accident")
    
    # Step 6: Drop helper columns from explode_df
    df_usager_exploded = df_usager_exploded.drop(columns=['nunique_lengths', 'max_length'])
    print("Step 6: Dropped helper columns from explode_df")
    
    # Step 7: Assign to global cleaned DataFrame
    df_usagers_cleaned = df_usager_exploded
    print("✅ df_usagers_cleaned transformation completed.\n")
    print(df_usagers_cleaned.info())


In [ ]:
transform_usagers()

---
## 3 - LOAD

### Création et connexion à la base de données

#### Fonction `create_database`
Cette fonction permet de **vérifier l'existence** d'une base de données PostgreSQL et de la **créer si nécessaire**.

##### Fonctionnement :
1. **Connexion initiale** :
   - Utilise les informations de connexion (`user`, `password`, `host`, `port`, `db_default`) pour se connecter à une base PostgreSQL par défaut (souvent `postgres`).

2. **Vérification de l'existence** :
   - Exécute une requête SQL pour vérifier si la base de données (`db_name` attention c'est une variable configurée dans votre config.yml local) existe déjà.

3. **Création si nécessaire** :
   - Si la base n'existe pas, elle est créée avec la commande `CREATE DATABASE`.

4. **Gestion des erreurs** :
   - Capture et affiche les erreurs potentielles (ex : problèmes de connexion, permissions).

5. **Fermeture de la connexion** :
   - Libère les ressources en fermant proprement la connexion à la base de données.

##### Objectif :
Préparer l'environnement pour les étapes suivantes de chargement des données nettoyées dans la base.


In [ ]:
def create_database(db_config: dict, db_name: str):
    """
    Vérifie si une base PostgreSQL existe et la crée si nécessaire.

    Args:
        db_config (dict): Dictionnaire contenant les informations de connexion :
            
user : nom d'utilisateur PostgreSQL
password : mot de passe
host : adresse du serveur
port : port PostgreSQL
db_default : base par défaut utilisée pour se connecter initialement
db_name (str): Nom de la base à créer ou vérifier.

    Returns:
        None
    """

    # Construction de l'URL de connexion sur la base par défaut (souvent 'postgres')
    db_url = (
        f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}"
        f"@{db_config['host']}:{db_config['port']}/{db_config['db_default']}"
    )
    try:
        # Création de l'engine avec autocommit pour exécuter CREATE DATABASE
        engine = create_engine(db_url, isolation_level="AUTOCOMMIT")

        with engine.connect() as conn:
            # Vérifier si la base existe déjà
            result = conn.execute(
                text("SELECT 1 FROM pg_database WHERE datname = :dbname"),
                {"dbname": db_name}
            )
            exists = result.scalar()  # Récupère le premier résultat (1 si la base existe)

            if not exists:
                # Crée la base si elle n'existe pas
                conn.execute(text(f'CREATE DATABASE "{db_name}"'))
                print(f"Base '{db_name}' créée.")
            else:
                print(f"Base '{db_name}' existe déjà.")
    except SQLAlchemyError as e:
        # Gestion des erreurs SQLAlchemy
        print(f"Erreur lors de la vérification ou création de la base : {e}")

    finally:
        # Fermeture propre de l'engine
        if 'engine' in locals():
            engine.dispose()

In [ ]:
create_database(db_conf, db_conf["db_name"])

---

### Chargement des données dans PostgreSQL

#### Fonction `load_df_to_postgres`
Cette fonction permet de **charger un DataFrame pandas dans une table PostgreSQL**.

##### Paramètres :
- **`db_conf`** : Dictionnaire contenant les informations de connexion à la base de données (`user`, `password`, `host`, `port`).
- **`db_name`** : Nom de la base de données cible.
- **`df`** : DataFrame pandas à insérer.
- **`table_name`** : Nom de la table cible dans PostgreSQL.
- **`if_exists`** : Comportement si la table existe déjà :
  - `'fail'` : Lève une erreur.
  - `'replace'` : Supprime et recrée la table.
  - `'append'` : Ajoute les lignes (par défaut).

##### Fonctionnement :
1. **Vérification du DataFrame** :
   - Si le DataFrame est vide, retourne un message d'avertissement.

2. **Connexion à la base** :
   - Construit l'URL de connexion à la base de données cible.

3. **Chargement des données** :
   - Utilise `to_sql()` pour insérer le DataFrame dans la table PostgreSQL.

4. **Gestion des erreurs** :
   - Capture et retourne les erreurs potentielles (ex : problèmes de connexion, permissions, contraintes de table).

5. **Fermeture de la connexion** :
   - Libère les ressources en fermant proprement la connexion à la base de données.

##### Résultat :
- Retourne un message de succès ou d'erreur, indiquant le nombre de lignes chargées ou la nature de l'erreur.

In [ ]:
def load_df_to_postgres(db_conf: dict, db_name: str, df: pd.DataFrame, table_name: str, if_exists: str = "append") -> str:
    """
    Charge un DataFrame dans une table PostgreSQL.

    Args:
        db_conf (dict): Informations de connexion :
            
user : nom d'utilisateur PostgreSQL
password : mot de passe
host : adresse du serveur
port : port PostgreSQL
db_name (str): Nom de la base de données cible.
df (pd.DataFrame): DataFrame à insérer.
table_name (str): Nom de la table cible dans PostgreSQL.
if_exists (str): Comportement si la table existe déjà :
'fail' : lève une erreur
'replace' : supprime et recrée la table
'append' : ajoute les lignes (par défaut)

    Returns:
        str: Message de succès ou d'erreur.
    """
    if df.empty:
        return "⚠️ Le DataFrame est vide, rien à charger dans la base."

    # Construction de l’URL de connexion PostgreSQL
    connection_url = (
        f"postgresql+psycopg2://{db_conf['user']}:{db_conf['password']}"
        f"@{db_conf['host']}:{db_conf['port']}/{db_name}"
    )

    try:
        engine = create_engine(connection_url)

        # Chargement du DataFrame dans la table PostgreSQL
        df.to_sql(table_name, engine, if_exists=if_exists, index=False)

        return f"✅ DataFrame chargé avec succès dans la table '{table_name}' ({len(df)} lignes)."

    except SQLAlchemyError as e:
        return f"❌ Erreur lors du chargement du DataFrame dans la base : {e}"

    finally:
        if 'engine' in locals():
            engine.dispose()

### Itération du chargement des données dans PostgreSQL

#### Processus :
Cette boucle permet de **charger successivement chaque DataFrame nettoyé** dans une table PostgreSQL dédiée.

##### Étapes :
1. **Dictionnaire des DataFrames** :
   - `cleaned_dfs` contient les DataFrames nettoyés, organisés par thème :
     - `accident`
     - `lieux`
     - `date_accident`
     - `vehicule`
     - `usager`

2. **Boucle de chargement** :
   - Pour chaque **nom** et **DataFrame** (`name`, `df`) dans `cleaned_dfs` :
     - Définit le nom de la table PostgreSQL (`table_name`) comme étant le même que le nom du DataFrame.
     - Appelle la fonction `load_df_to_postgres` avec les paramètres suivants :
       - **`db_conf`** : Configuration de la base de données.
       - **`db_name`** : Nom de la base de données cible.
       - **`df`** : DataFrame à charger.
       - **`table_name`** : Nom de la table cible.
       - **`if_exists="replace"`** : Remplace la table si elle existe déjà.
     - Affiche le **résultat** du chargement (succès ou erreur).

##### Objectif :
- **Automatiser** le chargement de tous les DataFrames nettoyés dans leurs tables respectives.
- **Garantir** que chaque table est recréée à chaque exécution, évitant ainsi les doublons ou incohérences.

In [ ]:
cleaned_dfs = {
    "accident": df_accidents_cleaned,
    "lieux": df_lieux_cleaned,
    "date_accident": df_date_accident_cleaned,
    "vehicule": df_vehicules_cleaned,
    "usager": df_usagers_cleaned
}


for name, df in cleaned_dfs.items():
    table_name = name  # optional naming convention in DB
    result = load_df_to_postgres(db_conf, db_conf['db_name'], df, table_name, if_exists="replace")
    print(result)

---

### Exécution de requêtes SQL sur PostgreSQL

#### Fonction `execute_sql_query`
Cette fonction permet d'**exécuter une requête SQL** sur une base de données PostgreSQL, **sans retourner de résultat** (idéal pour les requêtes de type `INSERT`, `UPDATE`, `DELETE`, `CREATE TABLE`, etc.).

##### Paramètres :
- **`db_conf`** : Dictionnaire contenant les informations de connexion (`user`, `password`, `host`, `port`).
- **`db_name`** : Nom de la base de données cible.
- **`sql_query`** : Chaîne de caractères représentant la requête SQL à exécuter.

##### Fonctionnement :
1. **Connexion à la base** :
   - Construit l'URL de connexion à la base de données cible.
   - Utilise `isolation_level="AUTOCOMMIT"` pour valider automatiquement les transactions.

2. **Exécution de la requête** :
   - Exécute la requête SQL fournie via `conn.execute(text(sql_query))`.

3. **Gestion des erreurs** :
   - Capture et retourne les erreurs potentielles (ex : syntaxe SQL invalide, problèmes de connexion).

4. **Fermeture de la connexion** :
   - Libère les ressources en fermant proprement la connexion à la base de données.

##### Résultat :
- Retourne un **message de succès** si la requête s'exécute correctement.
- Retourne un **message d'erreur** en cas d'échec, avec les détails de l'erreur.


In [ ]:
def execute_sql_query(db_conf: dict, db_name: str, sql_query: str) -> str:
    """
    Exécute une requête SQL (string) sur une base PostgreSQL sans retourner de DataFrame.
    Retourne uniquement un message de succès ou d'erreur.

    Args:
        db_conf (dict): Informations de connexion à la base PostgreSQL :
            
user : nom d'utilisateur
password : mot de passe
host : adresse du serveur
port : port PostgreSQL
db_name (str): Nom de la base de données cible.
sql_query (str): Requête SQL à exécuter.

    Returns:
        str: Message de succès ou d'erreur.
    """
    # Construction de l’URL de connexion PostgreSQL
    connection_url = (
        f"postgresql+psycopg2://{db_conf['user']}:{db_conf['password']}"
        f"@{db_conf['host']}:{db_conf['port']}/{db_name}"
    )

    try:
        engine = create_engine(connection_url, isolation_level="AUTOCOMMIT")

        with engine.connect() as conn:
            conn.execute(text(sql_query))
        return f"✅ Requête exécutée avec succès sur la base '{db_name}'."

    except SQLAlchemyError as e:
        return f"❌ Erreur lors de l'exécution de la requête : {e}"

    finally:
        if 'engine' in locals():
            engine.dispose()

### Ajout des contraintes SQL

**Objectif** : Appliquer des règles d'intégrité aux tables PostgreSQL.

#### Contraintes
| Table           | Clé primaire       | Clé étrangère (→ accident) | Action          |
|-----------------|--------------------|----------------------------|-----------------|
| `accident`      | `num_acc`          | -                          | -               |
| `lieux`         | `lieu_id`          | `num_acc`                  | `CASCADE`       |
| `date_accident` | `date_accident_id` | `num_acc`                  | `CASCADE`       |
| `vehicule`      | `vehicule_id`      | `num_acc`                  | `CASCADE`       |
| `usager`        | `usager_id`        | `num_acc`                  | `CASCADE`       |

**Exécution** : La requête est envoyée via `execute_sql_query`.

In [ ]:
# Adding constraints to database
query = """
-- SQL Constraints for accident database

-- Table: accident
ALTER TABLE accident ADD CONSTRAINT pk_accident PRIMARY KEY (num_acc);

-- Table: lieux
ALTER TABLE lieux ADD CONSTRAINT pk_lieux PRIMARY KEY (lieu_id);
ALTER TABLE lieux ADD CONSTRAINT fk_lieux_num_acc FOREIGN KEY (num_acc) REFERENCES accident(num_acc) ON DELETE CASCADE;

-- Table: date_accident
ALTER TABLE date_accident ADD CONSTRAINT pk_date_accident PRIMARY KEY (date_accident_id);
ALTER TABLE date_accident ADD CONSTRAINT fk_date_accident_num_acc FOREIGN KEY (num_acc) REFERENCES accident(num_acc) ON DELETE CASCADE;

-- Table: vehicule
ALTER TABLE vehicule ADD CONSTRAINT pk_vehicule PRIMARY KEY (vehicule_id);
ALTER TABLE vehicule ADD CONSTRAINT fk_vehicule_num_acc FOREIGN KEY (num_acc) REFERENCES accident(num_acc) ON DELETE CASCADE;

-- Table: usager
ALTER TABLE usager ADD CONSTRAINT pk_usager PRIMARY KEY (usager_id);
ALTER TABLE usager ADD CONSTRAINT fk_usager_num_acc FOREIGN KEY (num_acc) REFERENCES accident(num_acc) ON DELETE CASCADE;
"""

execute_sql_query(db_conf, db_conf['db_name'], query)